# Amazon ML Challenge 2026: Entity Resolution, full-data run (E001)

**Settings (right panel):** Accelerator = None (CPU is enough; 30 GB RAM) · Internet = **On** · Persistence = Files.
**Input:** add your private dataset containing `train/` and `test/` (and ideally `utils/validate_submission.py`).

Run all cells top to bottom. Every stage caches to `/kaggle/working/work`, so if the session dies you can
rerun and finished stages are skipped.

In [ ]:
# 1. Config
EXP        = "20260925-E001-lgb-v1"
CV_FRAC    = 0.3      # fraction of train Source-1 records used for CV / fit features
K_COMB     = 40       # top-K by name+address score per S1
K_NAME     = 10       # extra top-K by name score only
DF_CAP     = 2000     # drop blocking keys more frequent than this
REPO       = "https://github.com/Bexwane/AmazonMLchallenge.git"
CODE_DIR   = "/kaggle/working/ber"
WORK       = "/kaggle/working/work"
OUT        = "/kaggle/working/output"

In [ ]:
# 2. Locate the dataset and the official validator anywhere under /kaggle/input
import glob, os
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
assert hits, "Dataset not found: add the dataset with train/ and test/ folders as notebook input"
DATA = os.path.dirname(os.path.dirname(hits[0]))
assert os.path.exists(f"{DATA}/test/test_source1.tsv"), "test/ folder missing next to train/"
val = glob.glob("/kaggle/input/**/validate_submission.py", recursive=True)
VALIDATOR = val[0] if val else None
print("DATA =", DATA)
print("VALIDATOR =", VALIDATOR)
!ls -la {DATA}/train {DATA}/test
!free -g; nproc

In [ ]:
# 3. Code + dependencies (Kaggle already ships numpy/pandas/scipy/sklearn/lightgbm/pyarrow)
!rm -rf {CODE_DIR} && git clone -q {REPO} {CODE_DIR} && cd {CODE_DIR} && git log --oneline -1
!pip install -q rapidfuzz==3.14.6
import sys; sys.path.insert(0, f"{CODE_DIR}/src")
import rapidfuzz, lightgbm, pandas, numpy, sklearn
print(rapidfuzz.__version__, lightgbm.__version__, pandas.__version__, numpy.__version__, sklearn.__version__)

In [ ]:
# 4. Unit tests (metric + normalizer) must pass before spending compute
!cd {CODE_DIR} && python -m pytest -q tests

In [ ]:
# helper: run a pipeline stage with live log streaming
import subprocess, time
def stage(cmd, *extra):
    args = ["python", "-m", "ber.run", cmd, "--data", DATA, "--work", WORK, "--exp", EXP,
            "--cv-frac", str(CV_FRAC), "--k-comb", str(K_COMB), "--k-name", str(K_NAME),
            "--df-cap", str(DF_CAP), *extra]
    t = time.time()
    p = subprocess.Popen(args, cwd=CODE_DIR, env={**os.environ, "PYTHONPATH": "src"},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, f"{cmd} failed (exit {p.returncode})"
    print(f"--- {cmd} done in {(time.time() - t) / 60:.1f} min")

In [ ]:
# 5. CV: normalize train -> blocking -> features -> 5-fold OOF + country-holdout stress CV
# First run: normalization ~15-25 min, blocking ~20-40 min, features ~20 min, LightGBM ~20-40 min.
stage("cv")

In [ ]:
# 6. Read the CV report. STOP here and send cv_metrics.json if pair_recall < 0.95.
import json
r = json.load(open(f"{WORK}/experiments/{EXP}/cv_metrics.json"))
b = r["blocking"]
print("blocking:", {k: round(v, 4) for k, v in b.items() if isinstance(v, float)})
print("threshold:", r["threshold"])
for k, v in r.items():
    if isinstance(v, dict) and "macro_f05" in v:
        print(f"{k:24s} macroF05={v['macro_f05']:.4f}  P={v['micro_precision']:.4f}  R={v['micro_recall']:.4f}  "
              f"single={v['f05_singletons']:.4f}  multi={v['f05_nonsingletons']:.4f}")
print("top features:", list(r["feature_gain_top"].items())[:8])

In [ ]:
# 7. Fit the final model on the sampled train pairs
stage("fit")

In [ ]:
# 8. Predict test (normalize test -> blocking -> streamed features -> exclusive assignment + threshold)
stage("predict", "--out", OUT)

In [ ]:
# 9. Validate with the official script (falls back to built-in checks if it isn't in the inputs)
if VALIDATOR:
    !python {VALIDATOR} --matching {OUT}/matching_results.tsv --candidate {OUT}/candidate_pairs.tsv --test-dir {DATA}/test --check-ids
else:
    import pandas as pd, csv
    s1 = pd.read_csv(f"{DATA}/test/test_source1.tsv", sep="	", quoting=csv.QUOTE_NONE, dtype=str, usecols=[0])
    m = pd.read_csv(f"{OUT}/matching_results.tsv", sep="	", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
    assert list(m.columns) == ["source1_entity_id", "matched_entity_ids"]
    assert m["source1_entity_id"].is_unique and set(m["source1_entity_id"]) == set(s1["entity_id"])
    lists = m["matched_entity_ids"].str.split(",").map(lambda x: [i for i in x if i])
    assert all(len(l) == len(set(l)) and all(i.startswith(("S2-", "S3-")) for i in l) for l in lists)
    print("basic checks PASS (official validator not found in inputs)")

In [ ]:
# 10. Prediction summary + sanity by country
import pandas as pd, csv
m = pd.read_csv(f"{OUT}/matching_results.tsv", sep="	", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
s1 = pd.read_csv(f"{DATA}/test/test_source1.tsv", sep="	", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
m = m.merge(s1[["entity_id", "country"]], left_on="source1_entity_id", right_on="entity_id")
m["n"] = m["matched_entity_ids"].map(lambda s: len(s.split(",")) if s else 0)
print(m.groupby("country")["n"].agg(["mean", lambda x: (x == 0).mean()]).rename(columns={"<lambda_0>": "empty_rate"}))
print("train reference: mean matches/S1 = 3.46, singleton rate = 0.056")
!ls -la {OUT}; du -sh {WORK}

## After the run
1. Download `/kaggle/working/output/matching_results.tsv` → upload to the portal (submission #1).
2. Send back `work/experiments/<EXP>/cv_metrics.json`, the public LB score, and the table from cell 10.
3. Keep this notebook version (Save Version → "Save output") so `work/` caches can be reused as input later.